<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/BIS_LAB/blob/main/week31bm24cs115_hemachandranr_cuckoosearch_cloudcomputingvm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random
import scipy.special as special

# ==============================================================================
# 1. PROBLEM SPECIFICATIONS: CLOUD RESOURCE ALLOCATION
# Application: Task Scheduling onto Virtual Machines (VMs)
# ==============================================================================
num_tasks = 12
num_vms = 4

# Task definitions: [CPU demand (cores), Memory demand (GB), Task length (MI)]
tasks = {
    i: {
        "cpu": random.randint(1, 4),
        "mem": random.randint(2, 8),
        "length": random.randint(1000, 5000),
        "priority": random.randint(1, 3) # 1=High, 2=Medium, 3=Low
    } for i in range(num_tasks)
}

# VM definitions: [CPU capacity, Memory capacity, Cost per unit time, processing speed (MIPS)]
vms = {
    j: {
        "cpu_cap": 16,
        "mem_cap": 32,
        "cost_rate": [1.5, 2.5, 3.5, 5.0][j],
        "mips": [1000, 1500, 2000, 3000][j]
    } for j in range(num_vms)
}

# ==============================================================================
# 2. CUCKOO SEARCH ALGORITHM PARAMETERS
# ==============================================================================
num_nests = 15
max_iterations = 50
pa = 0.25         # Discovery probability of alien eggs
INF = float('inf')

# ==============================================================================
# 3. FITNESS / OBJECTIVE FUNCTION (Time + Cost + Penalty)
# ==============================================================================
def evaluate_allocation(nest):
    """
    Evaluates an allocation plan (nest).
    nest array index = task ID, value = assigned VM ID.
    """
    vm_cpu_usage = np.zeros(num_vms)
    vm_mem_usage = np.zeros(num_vms)
    vm_execution_times = np.zeros(num_vms)
    total_cost = 0

    # Calculate resource load and execution metrics
    for task_id, vm_id in enumerate(nest):
        vm_id = int(vm_id)
        vm_cpu_usage[vm_id] += tasks[task_id]["cpu"]
        vm_mem_usage[vm_id] += tasks[task_id]["mem"]

        # Time = Length / Speed
        task_time = tasks[task_id]["length"] / vms[vm_id]["mips"]
        # Prioritize high-priority tasks by virtually scaling down their perceived delay
        weighted_time = task_time * (tasks[task_id]["priority"] / 2.0)

        vm_execution_times[vm_id] += weighted_time
        total_cost += task_time * vms[vm_id]["cost_rate"]

    # Execution time of system is determined by the slowest VM (Makespan)
    makespan = np.max(vm_execution_times)

    # Check constraints and apply heavy penalties for capacity overloads
    penalty = 0
    for j in range(num_vms):
        if vm_cpu_usage[j] > vms[j]["cpu_cap"]:
            penalty += (vm_cpu_usage[j] - vms[j]["cpu_cap"]) * 5000
        if vm_mem_usage[j] > vms[j]["mem_cap"]:
            penalty += (vm_mem_usage[j] - vms[j]["mem_cap"]) * 5000

    # Objective: Minimize combined execution time, cost, and penalties
    return makespan + total_cost + penalty

# ==============================================================================
# HELPER: LÉVY FLIGHT STEP GENERATOR
# ==============================================================================
def levy_flight(Lambda=1.5):
    sigma = (special.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
             (special.gamma((1 + Lambda) / 2) * Lambda * 2 ** ((Lambda - 1) / 2))) ** (1 / Lambda)
    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)
    step = u / (abs(v) ** (1 / Lambda))
    return step

# ==============================================================================
# 4. INITIALIZE POPULATION
# ==============================================================================
# Generate random nests where each task is randomly mapped to a valid VM index
nests = np.random.randint(0, num_vms, size=(num_nests, num_tasks))
fitness = np.array([evaluate_allocation(nest) for nest in nests])

# Track global optimum allocation schedule
best_index = np.argmin(fitness)
global_best_nest = nests[best_index].copy()
global_best_fitness = fitness[best_index]

# ==============================================================================
# 5. MAIN CUCKOO SEARCH OPTIMIZATION LOOP
# ==============================================================================
print("Starting Cloud Resource Allocation Optimization via Cuckoo Search...\n")

for iteration in range(max_iterations):
    # --- Step A: Generate New Solution via Lévy Flight ---
    random_cuckoo_idx = random.randint(0, num_nests - 1)
    new_nest = nests[random_cuckoo_idx].copy()

    # Apply Lévy flight random walk to alter specific task allocations
    step_size = levy_flight()
    num_mutations = max(1, int(abs(step_size) % num_tasks))
    mutated_tasks = random.sample(range(num_tasks), num_mutations)

    for task_id in mutated_tasks:
        new_nest[task_id] = random.randint(0, num_vms - 1)

    # Evaluate new candidate allocation plan
    new_fitness = evaluate_allocation(new_nest)

    # Replace a random nest if the new plan performs better
    random_target_idx = random.randint(0, num_nests - 1)
    if new_fitness < fitness[random_target_idx]:
        nests[random_target_idx] = new_nest.copy()
        fitness[random_target_idx] = new_fitness

    # --- Step B: Discovery of Alien Eggs & Abandoning Worst Plans ---
    # Rank plans to identify worst allocations
    worst_indices = np.argsort(fitness)[-int(pa * num_nests):]

    # Replace a fraction (pa) of worst nests with brand new random setups
    for idx in worst_indices:
        nests[idx] = np.random.randint(0, num_vms, size=num_tasks)
        fitness[idx] = evaluate_allocation(nests[idx])

    # --- Step C: Update Global Best Plan ---
    current_best_idx = np.argmin(fitness)
    if fitness[current_best_idx] < global_best_fitness:
        global_best_fitness = fitness[current_best_idx]
        global_best_nest = nests[current_best_idx].copy()

# ==============================================================================
# 6. OUTPUT OPTIMAL SOLUTION DATA SHOWCASE
# ==============================================================================
print("=" * 65)
print("                     OPTIMIZATION COMPLETE")
print("=" * 65)
print("👉 Optimal Task Allocation Mapping Schedule:")
for t_id, vm_id in enumerate(global_best_nest):
    print(f"   • Task {t_id:02d} (Priority {tasks[t_id]['priority']}) ➔ Scheduled on VM {vm_id}")
print("-" * 65)

# Verify final resource load on optimized schedule
final_cpu_load = np.zeros(num_vms)
final_mem_load = np.zeros(num_vms)
for t_id, vm_id in enumerate(global_best_nest):
    final_cpu_load[vm_id] += tasks[t_id]["cpu"]
    final_mem_load[vm_id] += tasks[t_id]["mem"]

print("⚡ Virtual Machine Final Load Summary:")
for vm_id in range(num_vms):
    print(f"   🖥️  VM {vm_id} Load: CPU [{final_cpu_load[vm_id]}/{vms[vm_id]['cpu_cap']} Cores] | "
          f"Memory [{final_mem_load[vm_id]}/{vms[vm_id]['mem_cap']} GB]")
print("=" * 65)


Starting Cloud Resource Allocation Optimization via Cuckoo Search...

                     OPTIMIZATION COMPLETE
👉 Optimal Task Allocation Mapping Schedule:
   • Task 00 (Priority 3) ➔ Scheduled on VM 2
   • Task 01 (Priority 2) ➔ Scheduled on VM 1
   • Task 02 (Priority 1) ➔ Scheduled on VM 0
   • Task 03 (Priority 2) ➔ Scheduled on VM 1
   • Task 04 (Priority 2) ➔ Scheduled on VM 2
   • Task 05 (Priority 3) ➔ Scheduled on VM 3
   • Task 06 (Priority 1) ➔ Scheduled on VM 0
   • Task 07 (Priority 3) ➔ Scheduled on VM 3
   • Task 08 (Priority 3) ➔ Scheduled on VM 3
   • Task 09 (Priority 2) ➔ Scheduled on VM 2
   • Task 10 (Priority 2) ➔ Scheduled on VM 0
   • Task 11 (Priority 2) ➔ Scheduled on VM 3
-----------------------------------------------------------------
⚡ Virtual Machine Final Load Summary:
   🖥️  VM 0 Load: CPU [11.0/16 Cores] | Memory [18.0/32 GB]
   🖥️  VM 1 Load: CPU [5.0/16 Cores] | Memory [13.0/32 GB]
   🖥️  VM 2 Load: CPU [4.0/16 Cores] | Memory [19.0/32 GB]
   🖥️  VM